# RNN Text Completion — *Thus Spake Zarathustra*
**Goal:** train an RNN/LSTM to perform word-level text completion on *Thus Spake Zarathustra* (Project Gutenberg).  
This notebook assumes the file `zarathustra.txt` is already downloaded and placed in the same directory.

In [ ]:
# install dependencies, if necessary
#!pip install tensorflow nltk

In [2]:
# Imports
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.utils import to_categorical
from pathlib import Path
import nltk
from nltk.tokenize import word_tokenize

# Reproducibility
import random
random.seed(0)
np.random.seed(0)
tf.random.set_seed(0)

print("TensorFlow version:", tf.__version__)
nltk.download('punkt_tab')

TensorFlow version: 2.22.0-rc0


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Dhvanish\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

## Text Processing

This part of the code reads and processes the text into the array `tokens`.
You can just run this code without any modifications.

In [3]:
file_path = Path('zarathustra.txt')
if not file_path.exists():
    raise FileNotFoundError("zarathustra.txt not found. Please place the file in the notebook folder.")
text = file_path.read_text(encoding='utf-8')

tokens_list = word_tokenize(text.lower())
tokenizer = Tokenizer()
tokenizer.fit_on_texts([tokens_list])
total_words = len(tokenizer.word_index) + 1
tokens = tokenizer.texts_to_sequences([tokens_list])[0]

print('Total unique words:', total_words)
print('Total tokens (words):', len(tokens))

Total unique words: 8068
Total tokens (words): 111166


## Data Processing

The tokens should be processed into a format, that allows an RNN to learn patterns that allow it to predict the next word in a series.
Each sample should use the previous 10 words as input `X` and the next word as the target `y`.

Convert targets `y` to one-hot vectors, with each dimension corresponding to a distinct token.
If you use an Embedding layer (https://keras.io/api/layers/core_layers/embedding/) in the RNN later, `X` won't need additional encoding.

Split the data 80/10/10 for training, validation and testing.

In [4]:

# Data Processing

SEQ_LEN = 10

# Create samples: previous 10 words -> next word
sequences = []
targets = []

for i in range(SEQ_LEN, len(tokens)):
    sequences.append(tokens[i - SEQ_LEN:i])
    targets.append(tokens[i])

X = np.array(sequences, dtype=np.int32)
y = np.array(targets, dtype=np.int32)

# Split chronologically: 80% training, 10% validation, 10% testing
n = len(X)
train_end = int(n * 0.80)
val_end = int(n * 0.90)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print("Vocabulary size:", total_words)
print("Input shape:", X.shape)
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))
print("Example input:", X_train[0])
print("Example target:", y_train[0])


Vocabulary size: 8068
Input shape: (111156, 10)
Training samples: 88924
Validation samples: 11116
Testing samples: 11116
Example input: [3979   96   34    4  203  923    4   34   46   82]
Example target: 2068


## Model Training and Evaluation

Construct an RNN model, train it on the training data and evaluate its accuracy on the testing data.

Use the validation data for early stoppage after 5 epochs without improvements for the validation loss.

**Note**: Keep in mind to use an Embedding layer, if you haven't encoded `X`.

In [5]:

# Create batched datasets with one-hot target labels
BATCH_SIZE = 128

def make_dataset(X_data, y_data, shuffle=False):
    dataset = tf.data.Dataset.from_tensor_slices((X_data, y_data))

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=min(len(X_data), 10000),
            seed=0
        )

    dataset = dataset.batch(BATCH_SIZE)

    # Convert integer targets to one-hot vectors in each batch
    dataset = dataset.map(
        lambda x, y: (x, tf.one_hot(y, depth=total_words))
    )

    return dataset.prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(X_train, y_train, shuffle=True)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)

# Build the RNN model
model = Sequential([
    Embedding(
        input_dim=total_words,
        output_dim=64,
        input_length=SEQ_LEN
    ),
    SimpleRNN(128),
    Dense(total_words, activation="softmax")
])

model.summary()

# Compile the model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Stop if validation loss does not improve for 5 epochs
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

# Train the model
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[early_stopping]
)


c:\Users\Dhvanish\OneDrive\Desktop\AI & Robotics\Sem 2\Generative AI\Exercises\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:117: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30


c:\Users\Dhvanish\OneDrive\Desktop\AI & Robotics\Sem 2\Generative AI\Exercises\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


695/695 ━━━━━━━━━━━━━━━━━━━━ 36s 49ms/step - accuracy: 0.0747 - loss: 6.1964 - val_accuracy: 0.1025 - val_loss: 5.8714
Epoch 2/30
695/695 ━━━━━━━━━━━━━━━━━━━━ 32s 46ms/step - accuracy: 0.1191 - loss: 5.4592 - val_accuracy: 0.1212 - val_loss: 5.6720
Epoch 3/30
695/695 ━━━━━━━━━━━━━━━━━━━━ 32s 47ms/step - accuracy: 0.1420 - loss: 5.1227 - val_accuracy: 0.1315 - val_loss: 5.6104
Epoch 4/30
695/695 ━━━━━━━━━━━━━━━━━━━━ 32s 46ms/step - accuracy: 0.1621 - loss: 4.8728 - val_accuracy: 0.1378 - val_loss: 5.6306
Epoch 5/30
695/695 ━━━━━━━━━━━━━━━━━━━━ 32s 47ms/step - accuracy: 0.1792 - loss: 4.6515 - val_accuracy: 0.1402 - val_loss: 5.6190
Epoch 6/30
695/695 ━━━━━━━━━━━━━━━━━━━━ 34s 48ms/step - accuracy: 0.1967 - loss: 4.4464 - val_accuracy: 0.1429 - val_loss: 5.6481
Epoch 7/30
695/695 ━━━━━━━━━━━━━━━━━━━━ 33s 47ms/step - accuracy: 0.2098 - loss: 4.2589 - val_accuracy: 0.1453 - val_loss: 5.6846
Epoch 8/30
695/695 ━━━━━━━━━━━━━━━━━━━━ 33s 47ms/step - accuracy: 0.2260 - loss: 4.0843 - val_accurac

In [6]:

test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")


87/87 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.1451 - loss: 5.7313
Test Loss: 5.7313
Test Accuracy: 14.51%


## Text Generation

Finally, use the trained model to generate some sentences. For this purpose, start with the word "thus" and repeatedly let the model predict the next word.

**Note**: Don't just pick the token with the highest softmax value, but rather, treat the output of the model as a probability distribution for picking the next word.

In [ ]:
def generate_text(seed_text='thus', max_len=100):
    # dictionary mapping token indices to words
    index_word = {v: k for k, v in tokenizer.word_index.items()}
    generated = seed_text.lower().strip()

    for _ in range(max_len):
        token_list = tokenizer.texts_to_sequences([generated])[0]
        # pad sequences to SEQ_LEN, necessary for RNN input
        token_list = pad_sequences([token_list], maxlen=SEQ_LEN, padding='pre')

        # TODO: your code here



seed_texts = ['thus']
print("Example generated text:")

for seed in seed_texts:
    print(f"\nSeed: '{seed}'")
    for i in range(10):
        print(generate_text(seed, max_len=100))